# Team Omega - Machine Learning Project

In this project we will suggest a recomenders systems for books avalable in the universtity library. To do so, we have acces to a data base that gives us information of the user of which book they use.

This project focuses on the analysis of bibliographic and behavioral data within a library system. It leverages two primary datasets: a comprehensive catalog (items.csv) containing over 15,000 unique records—including titles, authors, publishers, and subject classifications—and a detailed interaction log (interactions_train.csv) documenting over 87,000 user activities mapped through precise timestamps. By integrating these files, we aim to explore user engagement patterns and develop a robust recommendation framework capable of delivering personalized content suggestions based on individual reading histories and document metadata.

In this project we will to type of recommenders systems, the user to user and the item to item approach. Moreover, we will try to find ways to improve this project for instance, by complementing or merging those approaches.

First, we will import the different libraries needed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
from google.colab import files
import gc

Then, we will import the data.

In [ ]:
url_1 = "https://raw.githubusercontent.com/Guido-wu/Machine_learning_Group_Omega/refs/heads/main/interactions_train.csv"
url_2 = "https://raw.githubusercontent.com/Guido-wu/Machine_learning_Group_Omega/refs/heads/main/items.csv"

interactions_train = pd.read_csv(url_1)
items = pd.read_csv(url_2)

In order to improve our recommenders system we will improve the quality of the data using an LLM. With the subject column, we will categoriezed the differents books to find paterns in the users behaviors.

In [ ]:
url_3 = "https://raw.githubusercontent.com/Guido-wu/Machine_learning_Group_Omega/refs/heads/main/items_with_categories.csv"

items_cat = pd.read_csv(url_3)

For the following project we will use the data from the second dataframe.

### Data Cleaning and preparation

First, we will sort by user and seperate the dataset in to dataframe. For each user, we will make a train dataset with the first 80% of books he read. Then the rest will be the test dataset used to compare and improve our model.

In [ ]:
#Sorting the data
user_books = interactions_train.merge(items_cat, on='i')
user_books = user_books.sort_values(["u", "t"])

We will now separate the data in two different data : Test and Train data.

This will help us to find which of the system is the most efficient for our recomenders model.

In [ ]:
# We rank by percentage the books reads by the users from the first book to the last book read.

user_books["pct_rank"] = user_books.groupby("u")["t"].rank(pct=True, method='dense')
user_books.reset_index(inplace=True, drop=True)

Now, we will divide the dataset in a train and test data:

In [ ]:
train_data = user_books[user_books["pct_rank"] < 0.8]
test_data = user_books[user_books["pct_rank"] >= 0.8]

##Creating the user - item matrix

The user item matrix is a binary matrix if one user has read or not one item. This matrix has a shape of n_users by n_items

In [ ]:
n_users = user_books.u.max() + 1
n_items = user_books.i.max() + 1

In [ ]:
def create_data_matrix(data, n_users, n_items):
    """
    This function returns a numpy matrix with shape (n_users, n_items).
    Each entry is a binary value indicating positive interaction.
    """
    data_matrix = np.zeros((n_users, n_items))
    data_matrix[data["u"].values, data["i"].values] = 1
    return data_matrix

In [ ]:
#Creating the train and test data
train_data_matrix = create_data_matrix(train_data, n_users, n_items)
test_data_matrix = create_data_matrix(test_data, n_users, n_items)
train_data_matrix = train_data_matrix.astype('float32')
test_data_matrix = test_data_matrix.astype('float32')


# Recomenders system

## Item to Item

We will create an item to item collaborative filtering. This means that we will look at the set of books that users liked.

First, we will compute the item to item similarity matrix:

In [ ]:
item_similarity = cosine_similarity(train_data_matrix.T)
item_similarity = item_similarity.astype('float32')

Now that we have the similarity matrix, we can compute the prediction and recall of our model with the split data mentionned before. First, we predict the likelihood of a positiv interaction.

In [ ]:
def item_based_predict(interactions, similarity, epsilon=1e-9):

    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon)
    return pred.T

item_prediction = item_based_predict(train_data_matrix, item_similarity)

## User to User

We will now do the user to user approach in order to extract the behavioral choices by users choosing a book.

In [ ]:
user_similarity = cosine_similarity(train_data_matrix)
user_similarity = user_similarity.astype('float32')

In [ ]:
def user_based_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions) / (np.abs(similarity).sum(axis=1)[:, np.newaxis] + epsilon)
    return pred

user_prediction = user_based_predict(train_data_matrix, user_similarity)

# First Comparision

Now that we have our both user to user and item to item train data, we can have a look at the differences between both of them. To do so, we will look at the precision @10 and the recall @10. Following those result, we will be able to built on this starting point to better our output.

The standards used:

- **Precision@K**: Measures the proportion of relevant items in the top-K recommendations.
  
  \begin{equation}
  \text{Precision@K} = \frac{\text{Number of relevant items in top-K}}{K}
  \end{equation}

- **Recall@K**: Measures the proportion of actual relevant items that appear in the top-K recommendations.

  \begin{equation}
  \text{Recall@K} = \frac{\text{Number of relevant items in top-K}}{\text{Total number of relevant items}}
  \end{equation}

Those standards will be generated with the following function :

In [ ]:
def precision_recall_at_k(prediction, ground_truth, k=10):
    """
    Calculates Precision@K and Recall@K for top-K recommendations.
    Parameters:
        prediction (numpy array): The predicted interaction matrix with scores.
        ground_truth (numpy array): The ground truth interaction matrix (binary).
        k (int): Number of top recommendations to consider.
    Returns:
        precision_at_k (float): The average precision@K over all users.
        recall_at_k (float): The average recall@K over all users.
    """
    num_users = prediction.shape[0]
    precision_at_k, recall_at_k = 0, 0

    for user in range(num_users):

        top_k_items = np.argsort(prediction[user])[::-1][:k]
        relevant_items_in_top_k = np.sum(ground_truth[user, top_k_items])

        total_relevant_items = np.sum(ground_truth[user])

        # Update Precision@K and Recall@K for this user
        precision_at_k += relevant_items_in_top_k / k
        recall_at_k += relevant_items_in_top_k / total_relevant_items if total_relevant_items > 0 else 0

    precision_at_k /= num_users
    recall_at_k /= num_users

    return precision_at_k, recall_at_k


In [ ]:
precision_user_k, recall_user_k = precision_recall_at_k(user_prediction, test_data_matrix, k=10)
precision_item_k, recall_item_k = precision_recall_at_k(item_prediction, test_data_matrix, k=10)

print('User-based CF Precision@K:', precision_user_k)
print('User-based CF Recall@K:', recall_user_k)
print('Item-based CF Precision@K:', precision_item_k)
print('Item-based CF Recall@K:', recall_item_k)

User-based CF Precision@K: 0.056559294
User-based CF Recall@K: 0.29067552
Item-based CF Precision@K: 0.05566608
Item-based CF Recall@K: 0.26398945


The results are the following :

$$
\begin{array}{|l|c|c|}
\hline
\text{Collaborative filtering model} & \text{Precision@10} & \text{Recall@10} \\
\hline
\text{User-based} & 0.0566 & 0.2907 \\
\text{Item-based} & 0.0557 & 0.2640 \\
\hline
\end{array}
$$

This means that, in this first basic model

# Expanding the model

To do so, we will first use the given data. We will try to integrate the genre, the authors and other information to link books to each other. We also need to add information in orther to help our recommender model with cold starts. This means when a book is added to the list or a new user join our platform. To do so, we will add content based approach.

With this approach, we will use the item to item collaborative filtering model combine with the categories and genre of the books.



The first step will be to create a cosine similarity matrix with the authors and items.

In [ ]:
# Defining the authors matrix - No need to separate the train and test data
all_item_ids = np.arange(n_items)
df_all_items = pd.DataFrame({'i': all_item_ids})

# Join ids on the authors dataframe
df_authors = user_books[['i', 'Author']].drop_duplicates('i')
df_final_content = pd.merge(df_all_items, df_authors, on='i', how='left')

# Fill the Nan values with Unknow_Author
df_final_content['Author'] = df_final_content['Author'].fillna('Unknown_Author')

# Putting the authors in a vector
vectorizer = CountVectorizer(tokenizer=lambda x: [x], lowercase=False)
author_matrix = vectorizer.fit_transform(df_final_content['Author'])

# Generating the similarity matrix :(n_items, n_items)
S_content = cosine_similarity(author_matrix)
S_content = S_content.astype('float32')

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


We will now prepare the matrix per category :

In [ ]:
all_item_ids = np.arange(n_items)
df_all_items = pd.DataFrame({'i': all_item_ids})

# Join ids on the categories dataframe
df_cat = user_books[['i', 'Catégorie']].drop_duplicates('i')
df_final_cat = pd.merge(df_all_items, df_cat, on='i', how='left')

# Fill the Nan values with Unknow_cat
df_final_cat['Catégorie'] = df_final_cat['Catégorie'].fillna('Unknown_cat')

# Putting the cat in a vector
vectorizer = CountVectorizer(tokenizer=lambda x: [x], lowercase=False)
cat_matrix = vectorizer.fit_transform(df_final_cat['Catégorie'])

# Generating the similarity matrix :(n_items, n_items)
S_cat = cosine_similarity(cat_matrix)
S_cat = S_cat.astype('float32')

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Now that we have our different matrice representing our books, we can find at which weight of importance those will be implemented. To do so, we will use the train data with the item to item collaborative filtering model. This code is time intensive, we ran it once and the results will be available in the next cell.

In [ ]:
alphas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
beta = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
results = []

#Generating item to item matrix with the user train data (This means just taking 80% of the data) (S_content don't change because independant from user)
'''
for b in beta:
    S_Hybrid = (b * S_cat) + ((1 - b) * S_content)
    for a in alphas:
        # Hybrid train similarity matrix
        S_hybrid = (a * item_similarity) + ((1 - a) * S_Hybrid)

        # Prediction
        current_prediction = train_data_matrix.dot(S_hybrid)

        # Prediction and recall with the previously made function
        prec, rec = precision_recall_at_k(current_prediction, test_data_matrix, k=10)

        results.append({'alpha': a, 'beta':b, 'precision': prec, 'recall': rec})
        print(f"Alpha: {a:.1f} / Beta: {b:.1f} | Recall@10: {rec:.4f} | Precision@10: {prec:.4f}")

        del S_hybrid
        del current_prediction
        gc.collect() # Force Colab to empty the RAM
  '''

'\nfor b in beta:\n    S_Hybrid = (b * S_cat) + ((1 - b) * S_content)\n    for a in alphas:\n        # Hybrid train similarity matrix\n        S_hybrid = (a * item_similarity) + ((1 - a) * S_Hybrid)\n\n        # Prediction\n        current_prediction = train_data_matrix.dot(S_hybrid)\n\n        # Prediction and recall with the previously made function\n        prec, rec = precision_recall_at_k(current_prediction, test_data_matrix, k=10)\n\n        results.append({\'alpha\': a, \'beta\':b, \'precision\': prec, \'recall\': rec})\n        print(f"Alpha: {a:.1f} / Beta: {b:.1f} | Recall@10: {rec:.4f} | Precision@10: {prec:.4f}")\n\n        del S_hybrid\n        del current_prediction\n        gc.collect() # Force Colab to empty the RAM\n  '

In [ ]:
Results_opt = [{'alpha': 0.0, 'beta': 0.0, 'precision': np.float32(0.020796197), 'recall': np.float32(0.13015775)}, {'alpha': 0.1, 'beta': 0.0, 'precision': np.float32(0.049873736), 'recall': np.float32(0.2639247)}, {'alpha': 0.2, 'beta': 0.0, 'precision': np.float32(0.052591335), 'recall': np.float32(0.2682042)}, {'alpha': 0.3, 'beta': 0.0, 'precision': np.float32(0.055487555), 'recall': np.float32(0.27454185)}, {'alpha': 0.4, 'beta': 0.0, 'precision': np.float32(0.057605464), 'recall': np.float32(0.2804238)}, {'alpha': 0.5, 'beta': 0.0, 'precision': np.float32(0.05894513), 'recall': np.float32(0.2844531)}, {'alpha': 0.6, 'beta': 0.0, 'precision': np.float32(0.059442714), 'recall': np.float32(0.28571773)}, {'alpha': 0.7, 'beta': 0.0, 'precision': np.float32(0.059953064), 'recall': np.float32(0.2873329)}, {'alpha': 0.8, 'beta': 0.0, 'precision': np.float32(0.059812702), 'recall': np.float32(0.2858314)}, {'alpha': 0.9, 'beta': 0.0, 'precision': np.float32(0.059838213), 'recall': np.float32(0.2858317)}, {'alpha': 1.0, 'beta': 0.0, 'precision': np.float32(0.059264064), 'recall': np.float32(0.28180093)}, {'alpha': 0.0, 'beta': 0.1, 'precision': np.float32(0.023258671), 'recall': np.float32(0.14437267)}, {'alpha': 0.1, 'beta': 0.1, 'precision': np.float32(0.048699886), 'recall': np.float32(0.25834715)}, {'alpha': 0.2, 'beta': 0.1, 'precision': np.float32(0.0526806), 'recall': np.float32(0.2671146)}, {'alpha': 0.3, 'beta': 0.1, 'precision': np.float32(0.05594683), 'recall': np.float32(0.27638394)}, {'alpha': 0.4, 'beta': 0.1, 'precision': np.float32(0.05819236), 'recall': np.float32(0.2827845)}, {'alpha': 0.5, 'beta': 0.1, 'precision': np.float32(0.059366167), 'recall': np.float32(0.28623983)}, {'alpha': 0.6, 'beta': 0.1, 'precision': np.float32(0.059927557), 'recall': np.float32(0.28803912)}, {'alpha': 0.7, 'beta': 0.1, 'precision': np.float32(0.06032305), 'recall': np.float32(0.28857946)}, {'alpha': 0.8, 'beta': 0.1, 'precision': np.float32(0.060131676), 'recall': np.float32(0.2873073)}, {'alpha': 0.9, 'beta': 0.1, 'precision': np.float32(0.06009339), 'recall': np.float32(0.28708038)}, {'alpha': 1.0, 'beta': 0.1, 'precision': np.float32(0.059264064), 'recall': np.float32(0.28180093)}, {'alpha': 0.0, 'beta': 0.2, 'precision': np.float32(0.023245914), 'recall': np.float32(0.14432032)}, {'alpha': 0.1, 'beta': 0.2, 'precision': np.float32(0.046683997), 'recall': np.float32(0.25103378)}, {'alpha': 0.2, 'beta': 0.2, 'precision': np.float32(0.052068185), 'recall': np.float32(0.2644583)}, {'alpha': 0.3, 'beta': 0.2, 'precision': np.float32(0.055653363), 'recall': np.float32(0.27406552)}, {'alpha': 0.4, 'beta': 0.2, 'precision': np.float32(0.058141313), 'recall': np.float32(0.28200385)}, {'alpha': 0.5, 'beta': 0.2, 'precision': np.float32(0.059455458), 'recall': np.float32(0.28630814)}, {'alpha': 0.6, 'beta': 0.2, 'precision': np.float32(0.059991326), 'recall': np.float32(0.2883618)}, {'alpha': 0.7, 'beta': 0.2, 'precision': np.float32(0.060463395), 'recall': np.float32(0.28988874)}, {'alpha': 0.8, 'beta': 0.2, 'precision': np.float32(0.060310304), 'recall': np.float32(0.2879124)}, {'alpha': 0.9, 'beta': 0.2, 'precision': np.float32(0.060118914), 'recall': np.float32(0.28697625)}, {'alpha': 1.0, 'beta': 0.2, 'precision': np.float32(0.059264064), 'recall': np.float32(0.28180093)}, {'alpha': 0.0, 'beta': 0.3, 'precision': np.float32(0.023054535), 'recall': np.float32(0.14449617)}, {'alpha': 0.1, 'beta': 0.3, 'precision': np.float32(0.045369845), 'recall': np.float32(0.24691375)}, {'alpha': 0.2, 'beta': 0.3, 'precision': np.float32(0.05141749), 'recall': np.float32(0.2616657)}, {'alpha': 0.3, 'beta': 0.3, 'precision': np.float32(0.055461988), 'recall': np.float32(0.27276388)}, {'alpha': 0.4, 'beta': 0.3, 'precision': np.float32(0.05809027), 'recall': np.float32(0.28087753)}, {'alpha': 0.5, 'beta': 0.3, 'precision': np.float32(0.05916201), 'recall': np.float32(0.28456715)}, {'alpha': 0.6, 'beta': 0.3, 'precision': np.float32(0.060118917), 'recall': np.float32(0.28880522)}, {'alpha': 0.7, 'beta': 0.3, 'precision': np.float32(0.060629256), 'recall': np.float32(0.29079106)}, {'alpha': 0.8, 'beta': 0.3, 'precision': np.float32(0.06027202), 'recall': np.float32(0.28780073)}, {'alpha': 0.9, 'beta': 0.3, 'precision': np.float32(0.060106155), 'recall': np.float32(0.2869291)}, {'alpha': 1.0, 'beta': 0.3, 'precision': np.float32(0.059264064), 'recall': np.float32(0.28180093)}, {'alpha': 0.0, 'beta': 0.4, 'precision': np.float32(0.022225197), 'recall': np.float32(0.14056693)}, {'alpha': 0.1, 'beta': 0.4, 'precision': np.float32(0.044655338), 'recall': np.float32(0.24475187)}, {'alpha': 0.2, 'beta': 0.4, 'precision': np.float32(0.05088161), 'recall': np.float32(0.25962898)}, {'alpha': 0.3, 'beta': 0.4, 'precision': np.float32(0.055564065), 'recall': np.float32(0.27324492)}, {'alpha': 0.4, 'beta': 0.4, 'precision': np.float32(0.057835102), 'recall': np.float32(0.27948415)}, {'alpha': 0.5, 'beta': 0.4, 'precision': np.float32(0.05905992), 'recall': np.float32(0.28345287)}, {'alpha': 0.6, 'beta': 0.4, 'precision': np.float32(0.05994029), 'recall': np.float32(0.28758442)}, {'alpha': 0.7, 'beta': 0.4, 'precision': np.float32(0.06041236), 'recall': np.float32(0.2893883)}, {'alpha': 0.8, 'beta': 0.4, 'precision': np.float32(0.060501687), 'recall': np.float32(0.28975472)}, {'alpha': 0.9, 'beta': 0.4, 'precision': np.float32(0.06019547), 'recall': np.float32(0.28768957)}, {'alpha': 1.0, 'beta': 0.4, 'precision': np.float32(0.059264064), 'recall': np.float32(0.28180093)}, {'alpha': 0.0, 'beta': 0.5, 'precision': np.float32(0.020196525), 'recall': np.float32(0.13131794)}, {'alpha': 0.1, 'beta': 0.5, 'precision': np.float32(0.043851547), 'recall': np.float32(0.24148087)}, {'alpha': 0.2, 'beta': 0.5, 'precision': np.float32(0.050422292), 'recall': np.float32(0.25684464)}, {'alpha': 0.3, 'beta': 0.5, 'precision': np.float32(0.054900628), 'recall': np.float32(0.26943722)}, {'alpha': 0.4, 'beta': 0.5, 'precision': np.float32(0.05757992), 'recall': np.float32(0.27793077)}, {'alpha': 0.5, 'beta': 0.5, 'precision': np.float32(0.058855794), 'recall': np.float32(0.28220212)}, {'alpha': 0.6, 'beta': 0.5, 'precision': np.float32(0.059774414), 'recall': np.float32(0.28629157)}, {'alpha': 0.7, 'beta': 0.5, 'precision': np.float32(0.06042513), 'recall': np.float32(0.28987932)}, {'alpha': 0.8, 'beta': 0.5, 'precision': np.float32(0.060539957), 'recall': np.float32(0.29015312)}, {'alpha': 0.9, 'beta': 0.5, 'precision': np.float32(0.060259264), 'recall': np.float32(0.2877663)}, {'alpha': 1.0, 'beta': 0.5, 'precision': np.float32(0.059264064), 'recall': np.float32(0.28180093)}, {'alpha': 0.0, 'beta': 0.6, 'precision': np.float32(0.019545823), 'recall': np.float32(0.12505762)}, {'alpha': 0.1, 'beta': 0.6, 'precision': np.float32(0.043532576), 'recall': np.float32(0.23858462)}, {'alpha': 0.2, 'beta': 0.6, 'precision': np.float32(0.0502947), 'recall': np.float32(0.25579304)}, {'alpha': 0.3, 'beta': 0.6, 'precision': np.float32(0.054683723), 'recall': np.float32(0.26828966)}, {'alpha': 0.4, 'beta': 0.6, 'precision': np.float32(0.057311993), 'recall': np.float32(0.27711317)}, {'alpha': 0.5, 'beta': 0.6, 'precision': np.float32(0.058804758), 'recall': np.float32(0.28194386)}, {'alpha': 0.6, 'beta': 0.6, 'precision': np.float32(0.059583034), 'recall': np.float32(0.2847549)}, {'alpha': 0.7, 'beta': 0.6, 'precision': np.float32(0.060182713), 'recall': np.float32(0.28832424)}, {'alpha': 0.8, 'beta': 0.6, 'precision': np.float32(0.060233753), 'recall': np.float32(0.28872955)}, {'alpha': 0.9, 'beta': 0.6, 'precision': np.float32(0.060169946), 'recall': np.float32(0.28761855)}, {'alpha': 1.0, 'beta': 0.6, 'precision': np.float32(0.059264064), 'recall': np.float32(0.28180093)}, {'alpha': 0.0, 'beta': 0.7, 'precision': np.float32(0.01940547), 'recall': np.float32(0.12502353)}, {'alpha': 0.1, 'beta': 0.7, 'precision': np.float32(0.043315686), 'recall': np.float32(0.23722686)}, {'alpha': 0.2, 'beta': 0.7, 'precision': np.float32(0.050167114), 'recall': np.float32(0.2543894)}, {'alpha': 0.3, 'beta': 0.7, 'precision': np.float32(0.054339238), 'recall': np.float32(0.26666403)}, {'alpha': 0.4, 'beta': 0.7, 'precision': np.float32(0.057171647), 'recall': np.float32(0.27698883)}, {'alpha': 0.5, 'beta': 0.7, 'precision': np.float32(0.05875372), 'recall': np.float32(0.28173843)}, {'alpha': 0.6, 'beta': 0.7, 'precision': np.float32(0.05945545), 'recall': np.float32(0.2839173)}, {'alpha': 0.7, 'beta': 0.7, 'precision': np.float32(0.06008063), 'recall': np.float32(0.28753158)}, {'alpha': 0.8, 'beta': 0.7, 'precision': np.float32(0.06023375), 'recall': np.float32(0.28889576)}, {'alpha': 0.9, 'beta': 0.7, 'precision': np.float32(0.060055118), 'recall': np.float32(0.28721976)}, {'alpha': 1.0, 'beta': 0.7, 'precision': np.float32(0.059264064), 'recall': np.float32(0.28180093)}, {'alpha': 0.0, 'beta': 0.8, 'precision': np.float32(0.019341672), 'recall': np.float32(0.12467225)}, {'alpha': 0.1, 'beta': 0.8, 'precision': np.float32(0.04386431), 'recall': np.float32(0.23839898)}, {'alpha': 0.2, 'beta': 0.8, 'precision': np.float32(0.049733303), 'recall': np.float32(0.25208935)}, {'alpha': 0.3, 'beta': 0.8, 'precision': np.float32(0.05403303), 'recall': np.float32(0.2658436)}, {'alpha': 0.4, 'beta': 0.8, 'precision': np.float32(0.056941986), 'recall': np.float32(0.2757214)}, {'alpha': 0.5, 'beta': 0.8, 'precision': np.float32(0.05856234), 'recall': np.float32(0.28100547)}, {'alpha': 0.6, 'beta': 0.8, 'precision': np.float32(0.05913649), 'recall': np.float32(0.2832288)}, {'alpha': 0.7, 'beta': 0.8, 'precision': np.float32(0.059812695), 'recall': np.float32(0.28633806)}, {'alpha': 0.8, 'beta': 0.8, 'precision': np.float32(0.060259268), 'recall': np.float32(0.28901175)}, {'alpha': 0.9, 'beta': 0.8, 'precision': np.float32(0.060118914), 'recall': np.float32(0.28787878)}, {'alpha': 1.0, 'beta': 0.8, 'precision': np.float32(0.059264064), 'recall': np.float32(0.28180093)}, {'alpha': 0.0, 'beta': 0.9, 'precision': np.float32(0.01929064), 'recall': np.float32(0.12485692)}, {'alpha': 0.1, 'beta': 0.9, 'precision': np.float32(0.045676056), 'recall': np.float32(0.24330135)}, {'alpha': 0.2, 'beta': 0.9, 'precision': np.float32(0.049605716), 'recall': np.float32(0.2525089)}, {'alpha': 0.3, 'beta': 0.9, 'precision': np.float32(0.05361198), 'recall': np.float32(0.2635708)}, {'alpha': 0.4, 'beta': 0.9, 'precision': np.float32(0.05632957), 'recall': np.float32(0.2731187)}, {'alpha': 0.5, 'beta': 0.9, 'precision': np.float32(0.05824338), 'recall': np.float32(0.2806523)}, {'alpha': 0.6, 'beta': 0.9, 'precision': np.float32(0.05909821), 'recall': np.float32(0.28355402)}, {'alpha': 0.7, 'beta': 0.9, 'precision': np.float32(0.059723396), 'recall': np.float32(0.28578123)}, {'alpha': 0.8, 'beta': 0.9, 'precision': np.float32(0.060169965), 'recall': np.float32(0.2884224)}, {'alpha': 0.9, 'beta': 0.9, 'precision': np.float32(0.060106155), 'recall': np.float32(0.28790712)}, {'alpha': 1.0, 'beta': 0.9, 'precision': np.float32(0.059264064), 'recall': np.float32(0.28180093)}, {'alpha': 0.0, 'beta': 1.0, 'precision': np.float32(0.0016075543), 'recall': np.float32(0.009051559)}, {'alpha': 0.1, 'beta': 1.0, 'precision': np.float32(0.04641605), 'recall': np.float32(0.24577582)}, {'alpha': 0.2, 'beta': 1.0, 'precision': np.float32(0.049120907), 'recall': np.float32(0.2504992)}, {'alpha': 0.3, 'beta': 1.0, 'precision': np.float32(0.05242542), 'recall': np.float32(0.2589199)}, {'alpha': 0.4, 'beta': 1.0, 'precision': np.float32(0.055296123), 'recall': np.float32(0.26897842)}, {'alpha': 0.5, 'beta': 1.0, 'precision': np.float32(0.057388548), 'recall': np.float32(0.27677354)}, {'alpha': 0.6, 'beta': 1.0, 'precision': np.float32(0.058766488), 'recall': np.float32(0.28118864)}, {'alpha': 0.7, 'beta': 1.0, 'precision': np.float32(0.059519254), 'recall': np.float32(0.28437367)}, {'alpha': 0.8, 'beta': 1.0, 'precision': np.float32(0.059965815), 'recall': np.float32(0.28681073)}, {'alpha': 0.9, 'beta': 1.0, 'precision': np.float32(0.060093395), 'recall': np.float32(0.28759974)}, {'alpha': 1.0, 'beta': 1.0, 'precision': np.float32(0.059264064), 'recall': np.float32(0.28180093)}]

df_results = pd.DataFrame(Results_opt)

df_results.head()

,alpha,beta,precision,recall
0,0.0,0.0,0.020796,0.130158
1,0.1,0.0,0.049874,0.263925
2,0.2,0.0,0.052591,0.268204
3,0.3,0.0,0.055488,0.274542
4,0.4,0.0,0.057605,0.280424


In [ ]:
best_row = df_results.loc[df_results['precision'].idxmax()]
print(best_row)

alpha        0.700000
beta         0.300000
precision    0.060629
recall       0.290791
Name: 40, dtype: float64


Now that we found an approach optimum solution of the weight that can be used for our hybrid model we will combined this model with the user to user approach. We also see an improvement compare to the basic collaborative filtering models used in the previous section. On the same basis, we want to find the different weights that can be used to have the best recommendations.

In [ ]:
# We use the following matrix
best_row = df_results.loc[df_results['precision'].idxmax()]

# Extraire alpha et beta
a = best_row['alpha']
b = best_row['beta']

S_Hybrid_opt = a * item_similarity + (1-a)*(b * S_cat + (1-b) * S_content)


In [ ]:
'''
alphas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
results = []


pred_user = user_similarity.dot(train_data_matrix).astype('float32')
pred_item = train_data_matrix.dot(S_Hybrid_opt).astype('float32')

for a in alphas:
    #mixt prediction
    current_prediction = pred_user * a
    current_prediction += (1 - a) * pred_item

    # Prediction and recall with the previously made function
    prec, rec = precision_recall_at_k(current_prediction, test_data_matrix, k=10)

    results.append({'alpha': a, 'precision': prec, 'recall': rec})
    print(f"Alpha: {a:.1f}| Recall@10: {rec:.4f} | Precision@10: {prec:.4f}")
    del current_prediction
    gc.collect()

del pred_user, pred_item
gc.collect()
'''

'\nalphas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]\nresults = []\n\n\npred_user = user_similarity.dot(train_data_matrix).astype(\'float32\')\npred_item = train_data_matrix.dot(S_Hybrid_opt).astype(\'float32\')\n\nfor a in alphas:\n    #mixt prediction\n    current_prediction = pred_user * a\n    current_prediction += (1 - a) * pred_item\n\n    # Prediction and recall with the previously made function\n    prec, rec = precision_recall_at_k(current_prediction, test_data_matrix, k=10)\n\n    results.append({\'alpha\': a, \'precision\': prec, \'recall\': rec})\n    print(f"Alpha: {a:.1f}| Recall@10: {rec:.4f} | Precision@10: {prec:.4f}")\n    del current_prediction\n    gc.collect()\n\ndel pred_user, pred_item\ngc.collect()\n'

In [ ]:

alphas = [0.7]
results = []


pred_user = user_similarity.dot(train_data_matrix).astype('float32')
pred_item = train_data_matrix.dot(S_Hybrid_opt).astype('float32')

for a in alphas:
    #mixt prediction
    current_prediction = pred_user * a
    current_prediction += (1 - a) * pred_item

    # Prediction and recall with the previously made function
    prec, rec = precision_recall_at_k(current_prediction, test_data_matrix, k=10)

    results.append({'alpha': a, 'precision': prec, 'recall': rec})
    print(f"Alpha: {a:.1f}| Recall@10: {rec:.4f} | Precision@10: {prec:.4f}")
    del current_prediction
    gc.collect()

del pred_user, pred_item
gc.collect()


Alpha: 0.7| Recall@10: 0.3016 | Precision@10: 0.0625


0

With this method we could again improve the precision of our model.

Finally, we will try to improve our model by the popularity of the book. This means, we will have a look at the number of times a book as been choosen by a user and in fact, increase the recommendation of this book compare to oder. However, this approach could increase the bias in the data. Indeed, it will be a reinforcing loop for books that have already a large audience. The regulator would be the amount of this book in the library.

In [ ]:
# Popularity of books
item_popularity = train_data.groupby('i').size()

# popoularity vector
popularity_vector = np.zeros(n_items, dtype='float32')
for item_id, count in item_popularity.items():
    popularity_vector[item_id] = count

# normalizytion - score between 0 et 1
pop_min, pop_max = popularity_vector.min(), popularity_vector.max()
popularity_vector_norm = (popularity_vector - pop_min) / (pop_max - pop_min + 1e-9)


We will now add the popularity bias to the prediction :

In [ ]:
#Creating the popularity function

def add_popularity_bias(prediction, popularity_vector, gamma=0.1):
    return (1 - gamma) * prediction + gamma * popularity_vector[np.newaxis, :]

In [ ]:

a_opt = 0.7

pred_user = user_similarity.dot(train_data_matrix).astype('float32')
pred_item = train_data_matrix.dot(S_Hybrid_opt).astype('float32')

S_opt = a_opt * pred_user + (1-a_opt) * pred_item


gammas = [0.1]
for gamma in gammas:
    pred_with_pop = add_popularity_bias(S_opt, popularity_vector_norm, gamma=gamma)
    prec, rec = precision_recall_at_k(pred_with_pop, test_data_matrix, k=10)
    print(f"Gamma: {gamma:.2f} | Precision@10: {prec:.4f} | Recall@10: {rec:.4f}")



Gamma: 0.10 | Precision@10: 0.0625 | Recall@10: 0.3019


Now, we have found a possible optimum solution of our model. We can now impose this model to the entire dataset.

In [ ]:
final_data_matrix = create_data_matrix(user_books, n_users, n_items).astype('float32')

# Similarity matrix on the entire dataset
final_user_similarity = cosine_similarity(final_data_matrix).astype('float32')
final_item_similarity = cosine_similarity(final_data_matrix.T).astype('float32')

In [ ]:
#Popularity function on all datas

item_popularity_final = user_books.groupby('i').size()
popularity_vector_final = np.zeros(n_items, dtype='float32')
for item_id, count in item_popularity_final.items():
    popularity_vector_final[item_id] = count
pop_min, pop_max = popularity_vector_final.min(), popularity_vector_final.max()
popularity_vector_final_norm = (popularity_vector_final - pop_min) / (pop_max - pop_min + 1e-9)


In [ ]:
S_cat *= b
S_content *= (1 - b)
S_cat += S_content
del S_content
gc.collect()

S_cat *= (1 - a)
S_hybrid_final = S_cat
del S_cat
gc.collect()

0

In [ ]:
#User to item parameter
a_opt   = 0.7
#Popularity parameter
gamma   = 0.1

print('****')
final_item_similarity *= a
final_item_similarity += S_hybrid_final
del S_hybrid_final
gc.collect()


final_pred_item = final_data_matrix.dot(final_item_similarity).astype('float32')
del final_item_similarity
gc.collect()


final_pred_user = final_user_similarity.dot(final_data_matrix).astype('float32')
del final_user_similarity, final_data_matrix
gc.collect()


final_pred_user *= a_opt
final_pred_user += (1 - a_opt) * final_pred_item
del final_pred_item
gc.collect()


final_prediction = final_pred_user

#Final prediction

final_prediction = add_popularity_bias(final_prediction, popularity_vector_final_norm, gamma=gamma)



****


Generating the final recommendation csv to submit on kaggle.

In [ ]:
def generate_recommendation_csv(prediction_matrix, k=10, filename="recommendations.csv"):
    """
    Generate the top k recommendation for the user

    """
    num_users = prediction_matrix.shape[0]
    recommendations_list = []

    for user_id in range(num_users):
        # Top k best scores
        top_k_indices = np.argsort(prediction_matrix[user_id])[::-1][:k]

        rec_string = " ".join(map(str, top_k_indices))

        recommendations_list.append({
            'user_id': user_id,
            'recommendation': rec_string
        })

    # create a dataframe to export the submission file
    df_recos = pd.DataFrame(recommendations_list)
    df_recos.to_csv(filename, index=False)
    return df_recos


In [ ]:
df_final = generate_recommendation_csv(final_prediction, k=10, filename="final_submission.csv")
files.download('final_submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>